# Generación de instancias sintéticas

En este apartado se generan **10 instancias sintéticas** en formato CSV. Cada instancia representa un conjunto de equipos con dos atributos: si juega competiciones europeas y el estadio que utiliza. La semilla aleatoria se fija en `42` para que la generación pueda reproducirse.

## Tamaño de las instancias

Se generan instancias de **5 tamaños distintos**: 8, 10, 12, 14 y 16 equipos. Para cada tamaño se crean 2 versiones, por lo que hay 10 archivos en total. El tamaño se refleja en el nombre del archivo (por ejemplo, `Instancia_Sintetica_8Eq_v1.csv` corresponde a una instancia de 8 equipos).

## Selección de equipos que juegan en Europa

Para cada instancia, primero se crea la lista de equipos (`Equipo_1`, `Equipo_2`, etc.). Luego se elige aleatoriamente cuántos jugarán competiciones europeas: entre 2 y el mayor valor entre 2 y el 35 % del tamaño de la instancia (redondeado hacia abajo). Después se seleccionan esa cantidad de equipos **sin repetir**. La columna `Juega_Europa` toma el valor `1` para los elegidos y `0` para los demás.

## Equipos que comparten estadio

Cada instancia puede tener entre 0 y 2 pares de equipos que comparten estadio. Si se generan pares, se seleccionan al azar equipos distintos y se asigna a cada pareja un mismo identificador (`Estadio_Compartido_1` o `Estadio_Compartido_2`). Los equipos no seleccionados conservan su estadio individual (`Estadio_Equipo`). Esta selección es independiente de la condición de jugar en Europa.

In [6]:
import pandas as pd
import random
import os

# Configuración de las instancias
tamanos_instancias = [8, 10, 12, 14, 16]
instancias_por_tamano = 2

# Ruta de salida en OneDrive
ruta_salida = r"C:\Users\camip\OneDrive\Escritorio\580325 - Entrega 2 - Grupo 21\entrega2\INSTANCIAS"
os.makedirs(ruta_salida, exist_ok=True)

# Semilla opcional para reproducibilidad
random.seed(42)

for n in tamanos_instancias:
    for i in range(1, instancias_por_tamano + 1):
        equipos = [f"Equipo_{k}" for k in range(1, n + 1)]
        
        num_equipos_otras_ligas = random.randint(2, max(2, int(n * 0.35)))
        equipos_otras_ligas = set(random.sample(equipos, num_equipos_otras_ligas))
        
        # 1. Definir cantidad de equipos por nivel de audiencia/popularidad
        num_alta_audiencia = random.randint(2, max(2, n // 3))
        num_baja_audiencia = random.randint(2, max(2, n // 2))
        
        # 2. Seleccionar aleatoriamente los equipos para cada segmento de audiencia
        equipos_mezclados = random.sample(equipos, len(equipos))
        equipos_alta = set(equipos_mezclados[:num_alta_audiencia])
        equipos_baja = set(equipos_mezclados[num_alta_audiencia:num_alta_audiencia + num_baja_audiencia])
        
        datos = []
        for eq in equipos:
            # 3. Asignar el peso reflejando la popularidad
            if eq in equipos_alta:
                peso = 100  # Muy conocidos / Alta audiencia
            elif eq in equipos_baja:
                peso = 10   # Menos conocidos / Baja audiencia
            else:
                peso = 50   # Audiencia media
                
            datos.append({
                "Equipo": eq,
                "Juega_Otras_Ligas": 1 if eq in equipos_otras_ligas else 0,
                "ID_Estadio": f"Estadio_{eq}",
                "Peso_Audiencia": peso
            })
        
        df_instancia = pd.DataFrame(datos)
        
        num_pares_estadio = random.randint(0, 2)
        if num_pares_estadio > 0:
            pares_elegidos = random.sample(equipos, num_pares_estadio * 2)
            for j in range(num_pares_estadio):
                eq1 = pares_elegidos[2*j]
                eq2 = pares_elegidos[2*j + 1]
                estadio_comun = f"Estadio_Compartido_{j+1}"
                
                df_instancia.loc[df_instancia['Equipo'] == eq1, 'ID_Estadio'] = estadio_comun
                df_instancia.loc[df_instancia['Equipo'] == eq2, 'ID_Estadio'] = estadio_comun
                
        nombre_archivo = f"Instancia_Sintetica_{n}Eq_v{i}.csv"
        df_instancia.to_csv(os.path.join(ruta_salida, nombre_archivo), index=False)

print(f"Las 10 instancias sintéticas han sido generadas en: {ruta_salida}")

Las 10 instancias sintéticas han sido generadas en: C:\Users\camip\OneDrive\Escritorio\580325 - Entrega 2 - Grupo 21\entrega2\INSTANCIAS
